# Model Comparison

Compare all four sentiment classification approaches:
1. **Logistic Regression** with TF-IDF (baseline)
2. **Random Forest** with TF-IDF
3. **XGBoost** with TF-IDF
4. **LSTM Neural Network** with Keras tokenizer

This notebook evaluates all models on the test split and generates comprehensive comparison visualizations.

## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import pickle
import joblib
import numpy as np
import pandas as pd
import boto3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, auc
)
from sklearn.preprocessing import label_binarize
import torch
import torch.nn as nn

print(f'✓ Imports complete')

## Helper Classes

In [ ]:
class ModelComparison:
    """Compare all four sentiment classification models on the same test set."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3', region_name='us-east-1')
        self.df_test = None
        self.y_test = None
        self.int_max_len = 200
        self.list_model_names = ['Logistic Regression', 'Random Forest', 'XGBoost', 'LSTM']
        self.dict_predictions = {}
        self.dict_metrics = {}
        sns.set_style('whitegrid')
    
    def load_test_data(self):
        """Load test split from S3."""
        str_uri = f's3://{self.str_bucket}/02_preprocessing/test_split.parquet'
        self.df_test = pd.read_parquet(str_uri)
        self.y_test = self.df_test['sentiment_3class'].values
        print(f'Loaded test data from S3: {len(self.df_test)} samples')
    
    def load_tfidf_model(self, str_model_dir, str_model_filename):
        """Load a TF-IDF based model and generate predictions on test data."""
        str_vectorizer_path = os.path.join(str_model_dir, 'output', 'tfidf_vectorizer.pkl')
        str_model_path = os.path.join(str_model_dir, 'output', str_model_filename)
        
        vectorizer = joblib.load(str_vectorizer_path)
        model = joblib.load(str_model_path)
        
        X_test_tfidf = vectorizer.transform(self.df_test['review_text_clean'].values)
        y_pred = model.predict(X_test_tfidf)
        y_proba = model.predict_proba(X_test_tfidf)
        
        return y_pred, y_proba
    
    def load_lstm_model(self, str_model_dir):
        """Load LSTM neural network model using PyTorch."""
        
        class SentimentLSTM(nn.Module):
            """PyTorch LSTM model for sentiment classification."""
            def __init__(self, int_vocab_size, int_embedding_dim, int_lstm_units, int_num_classes, flt_dropout):
                super().__init__()
                self.embedding = nn.Embedding(int_vocab_size, int_embedding_dim, padding_idx=0)
                self.spatial_dropout = nn.Dropout2d(flt_dropout)
                self.lstm = nn.LSTM(int_embedding_dim, int_lstm_units, batch_first=True, dropout=flt_dropout)
                self.fc1 = nn.Linear(int_lstm_units, 32)
                self.relu = nn.ReLU()
                self.dropout = nn.Dropout(flt_dropout)
                self.fc2 = nn.Linear(32, int_num_classes)
            
            def forward(self, x):
                embedded = self.embedding(x)
                embedded = self.spatial_dropout(embedded.unsqueeze(3)).squeeze(3)
                lstm_out, (hidden, cell) = self.lstm(embedded)
                out = self.fc1(hidden[-1])
                out = self.relu(out)
                out = self.dropout(out)
                out = self.fc2(out)
                return out
        
        # Load vocabulary
        str_vocab_path = os.path.join(str_model_dir, 'output', 'vocabulary.pkl')
        dict_word2idx = joblib.load(str_vocab_path)
        
        # Load model config
        str_config_path = os.path.join(str_model_dir, 'output', 'model_config.pkl')
        dict_config = joblib.load(str_config_path)
        
        # Load model state dict
        str_model_path = os.path.join(str_model_dir, 'output', 'lstm_model.pt')
        
        # Rebuild model
        model = SentimentLSTM(
            int_vocab_size=dict_config['int_vocab_size'],
            int_embedding_dim=dict_config['int_embedding_dim'],
            int_lstm_units=dict_config['int_lstm_units'],
            int_num_classes=dict_config['int_num_classes'],
            flt_dropout=dict_config['flt_dropout']
        )
        model.load_state_dict(torch.load(str_model_path, map_location='cpu'))
        model.eval()
        
        # Convert text to sequences using vocabulary
        def texts_to_sequences(texts, dict_vocab, max_len):
            sequences = []
            for text in texts:
                words = text.lower().split()
                seq = [dict_vocab.get(word, 1) for word in words]
                if len(seq) < max_len:
                    seq.extend([0] * (max_len - len(seq)))
                else:
                    seq = seq[:max_len]
                sequences.append(seq)
            return np.array(sequences)
        
        # Prepare test data
        X_test = texts_to_sequences(
            self.df_test['review_text_clean'].values,
            dict_word2idx,
            dict_config['int_max_len']
        )
        
        # Convert to torch tensors and get predictions
        X_test_tensor = torch.from_numpy(X_test).long()
        
        with torch.no_grad():
            outputs = model(X_test_tensor)
            y_proba = torch.nn.functional.softmax(outputs, dim=1).numpy()
            y_pred = np.argmax(y_proba, axis=1)
        
        return y_pred, y_proba
    
    def compute_metrics(self, y_true, y_pred, y_proba):
        """Compute evaluation metrics for a single model."""
        y_bin = label_binarize(y_true, classes=[0, 1, 2])
        
        dict_metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'f1_macro': f1_score(y_true, y_pred, average='macro'),
            'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
            'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'roc_auc_macro': roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
        }
        
        # Per-class F1
        list_f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
        dict_metrics['f1_negative'] = list_f1_per_class[0]
        dict_metrics['f1_neutral'] = list_f1_per_class[1]
        dict_metrics['f1_positive'] = list_f1_per_class[2]
        
        return dict_metrics
    
    def generate_summary_table(self):
        """Print metrics summary table."""
        list_metric_names = ['accuracy', 'f1_macro', 'f1_weighted', 'roc_auc_macro', 'precision_macro', 'recall_macro']
        
        print('\n' + '='*90)
        print('MODEL COMPARISON SUMMARY')
        print('='*90)
        print(f'{"Metric":<20}', end='')
        for str_name in self.list_model_names:
            print(f'{str_name:<18}', end='')
        print()
        print('-'*90)
        
        for str_metric in list_metric_names:
            print(f'{str_metric:<20}', end='')
            list_values = []
            for int_i in range(4):
                flt_val = self.dict_metrics[int_i][str_metric]
                list_values.append(flt_val)
                print(f'{flt_val:<18.4f}', end='')
            
            # Mark best
            int_best = np.argmax(list_values)
            print(f'  <- {self.list_model_names[int_best]}')
        print('='*90)
    
    def plot_metrics_comparison(self):
        """Bar chart comparing key metrics across models."""
        list_metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'roc_auc_macro']
        list_labels = ['Accuracy', 'F1-Macro', 'F1-Weighted', 'ROC AUC']
        
        arr_x = np.arange(len(list_metrics))
        flt_width = 0.2
        list_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
        
        fig, ax = plt.subplots(figsize=(14, 6))
        
        for int_i, str_model in enumerate(self.list_model_names):
            list_values = [self.dict_metrics[int_i][m] for m in list_metrics]
            ax.bar(arr_x + int_i * flt_width, list_values, flt_width, label=str_model, 
                   color=list_colors[int_i], edgecolor='black')
        
        ax.set_xlabel('Metric', fontsize=12, fontweight='bold')
        ax.set_ylabel('Score', fontsize=12, fontweight='bold')
        ax.set_title('Model Comparison - Key Metrics', fontsize=14, fontweight='bold')
        ax.set_xticks(arr_x + flt_width * 1.5)
        ax.set_xticklabels(list_labels, fontsize=11)
        ax.legend(fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_metrics_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 01_metrics_comparison.png')
    
    def plot_roc_comparison(self):
        """Overlay ROC curves for all models."""
        list_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
        list_line_styles = ['-', '--', '-.', ':']
        y_bin = label_binarize(self.y_test, classes=[0, 1, 2])
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        list_class_names = ['Negative', 'Neutral', 'Positive']
        
        for int_cls in range(3):
            for int_i, str_model in enumerate(self.list_model_names):
                y_proba = self.dict_predictions[int_i]['y_proba']
                fpr, tpr, _ = roc_curve(y_bin[:, int_cls], y_proba[:, int_cls])
                flt_auc = auc(fpr, tpr)
                axes[int_cls].plot(fpr, tpr, color=list_colors[int_i], lw=2, 
                                   linestyle=list_line_styles[int_i],
                                   label=f'{str_model} (AUC={flt_auc:.3f})')
            
            axes[int_cls].plot([0, 1], [0, 1], 'k--', lw=1)
            axes[int_cls].set_title(f'{list_class_names[int_cls]} (One-vs-Rest)', fontsize=12, fontweight='bold')
            axes[int_cls].set_xlabel('False Positive Rate')
            axes[int_cls].set_ylabel('True Positive Rate')
            axes[int_cls].legend(fontsize=8, loc='lower right')
            axes[int_cls].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_roc_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 02_roc_comparison.png')
    
    def plot_confusion_comparison(self):
        """Side-by-side confusion matrices for all models."""
        fig, axes = plt.subplots(1, 4, figsize=(20, 4))
        list_class_names = ['Negative', 'Neutral', 'Positive']
        
        for int_i, str_model in enumerate(self.list_model_names):
            y_pred = self.dict_predictions[int_i]['y_pred']
            cm = confusion_matrix(self.y_test, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[int_i],
                       xticklabels=list_class_names, yticklabels=list_class_names)
            axes[int_i].set_title(str_model, fontsize=11, fontweight='bold')
            axes[int_i].set_xlabel('Predicted')
            axes[int_i].set_ylabel('True')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_confusion_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 03_confusion_comparison.png')
    
    def plot_per_class_f1(self):
        """Per-class F1 comparison across models."""
        list_classes = ['Negative', 'Neutral', 'Positive']
        list_f1_keys = ['f1_negative', 'f1_neutral', 'f1_positive']
        
        arr_x = np.arange(len(list_classes))
        flt_width = 0.2
        list_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        for int_i, str_model in enumerate(self.list_model_names):
            list_values = [self.dict_metrics[int_i][k] for k in list_f1_keys]
            ax.bar(arr_x + int_i * flt_width, list_values, flt_width, label=str_model,
                   color=list_colors[int_i], edgecolor='black')
        
        ax.set_xlabel('Sentiment Class', fontsize=12, fontweight='bold')
        ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
        ax.set_title('Per-Class F1 Score Comparison', fontsize=14, fontweight='bold')
        ax.set_xticks(arr_x + flt_width * 1.5)
        ax.set_xticklabels(list_classes, fontsize=11)
        ax.legend(fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_per_class_f1.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 04_per_class_f1.png')
    
    def model_disagreement_analysis(self):
        """Analyze where models disagree."""
        arr_all_preds = np.column_stack([self.dict_predictions[i]['y_pred'] for i in range(4)])
        
        # Count agreements
        arr_unanimous = np.all(arr_all_preds == arr_all_preds[:, 0:1], axis=1)
        int_unanimous = arr_unanimous.sum()
        int_total = len(arr_unanimous)
        
        print(f'\nModel Agreement Analysis')
        print(f'='*60)
        print(f'Total test samples: {int_total}')
        print(f'All 4 models agree: {int_unanimous} ({int_unanimous/int_total*100:.1f}%)')
        print(f'At least 1 disagrees: {int_total - int_unanimous} ({(int_total - int_unanimous)/int_total*100:.1f}%)')
        
        # Pairwise agreement
        print(f'\nPairwise Agreement Rates:')
        for int_i in range(4):
            for int_j in range(int_i + 1, 4):
                flt_agree = (arr_all_preds[:, int_i] == arr_all_preds[:, int_j]).mean()
                print(f'  {self.list_model_names[int_i]} vs {self.list_model_names[int_j]}: {flt_agree:.4f}')

print('ModelComparison class defined')

## Constants

In [ ]:
str_bucket = 'nlp-sentiment-analysis-demo'
str_dirname_output = './output'

# Model directory paths (relative to this notebook)
str_dir_logreg = '../03_tfidf_logreg'
str_dir_rf = '../04_tfidf_random_forest'
str_dir_xgb = '../05_tfidf_xgboost'
str_dir_lstm = '../06_neural_network'

list_model_dirs = [str_dir_logreg, str_dir_rf, str_dir_xgb, str_dir_lstm]
list_model_file_names = ['logreg_model.pkl', 'rf_model.pkl', 'xgboost_model.pkl', None]

print(f'✓ Constants defined')

## Output Directory

In [ ]:
os.makedirs(str_dirname_output, exist_ok=True)
print(f'✓ Output directory ready: {str_dirname_output}')

## Initialize Comparison

In [ ]:
cls_comparison = ModelComparison(str_bucket, str_dirname_output)
print(f'✓ ModelComparison initialized')

## Load Test Data

In [ ]:
cls_comparison.load_test_data()
print(f'✓ Test data loaded: {len(cls_comparison.df_test)} samples')
print(f'  Sentiment distribution:')
print(cls_comparison.df_test['sentiment_3class'].value_counts())

## Load All Models and Generate Predictions

In [ ]:
# Load Logistic Regression
print(f'Loading Logistic Regression...')
y_pred_lr, y_proba_lr = cls_comparison.load_tfidf_model(str_dir_logreg, 'logreg_model.pkl')
cls_comparison.dict_predictions[0] = {'y_pred': y_pred_lr, 'y_proba': y_proba_lr}
dict_metrics_lr = cls_comparison.compute_metrics(cls_comparison.y_test, y_pred_lr, y_proba_lr)
cls_comparison.dict_metrics[0] = dict_metrics_lr
print(f'✓ Logistic Regression loaded')

# Load Random Forest
print(f'Loading Random Forest...')
y_pred_rf, y_proba_rf = cls_comparison.load_tfidf_model(str_dir_rf, 'rf_model.pkl')
cls_comparison.dict_predictions[1] = {'y_pred': y_pred_rf, 'y_proba': y_proba_rf}
dict_metrics_rf = cls_comparison.compute_metrics(cls_comparison.y_test, y_pred_rf, y_proba_rf)
cls_comparison.dict_metrics[1] = dict_metrics_rf
print(f'✓ Random Forest loaded')

# Load XGBoost
print(f'Loading XGBoost...')
y_pred_xgb, y_proba_xgb = cls_comparison.load_tfidf_model(str_dir_xgb, 'xgboost_model.pkl')
cls_comparison.dict_predictions[2] = {'y_pred': y_pred_xgb, 'y_proba': y_proba_xgb}
dict_metrics_xgb = cls_comparison.compute_metrics(cls_comparison.y_test, y_pred_xgb, y_proba_xgb)
cls_comparison.dict_metrics[2] = dict_metrics_xgb
print(f'✓ XGBoost loaded')

# Load LSTM
print(f'Loading LSTM Neural Network...')
y_pred_lstm, y_proba_lstm = cls_comparison.load_lstm_model(str_dir_lstm)
cls_comparison.dict_predictions[3] = {'y_pred': y_pred_lstm, 'y_proba': y_proba_lstm}
dict_metrics_lstm = cls_comparison.compute_metrics(cls_comparison.y_test, y_pred_lstm, y_proba_lstm)
cls_comparison.dict_metrics[3] = dict_metrics_lstm
print(f'✓ LSTM Neural Network loaded')

print(f'\n✓ All models loaded successfully')

## Metrics Summary

In [ ]:
cls_comparison.generate_summary_table()

## Metrics Comparison

In [ ]:
cls_comparison.plot_metrics_comparison()

## ROC Curve Comparison

In [ ]:
cls_comparison.plot_roc_comparison()

## Confusion Matrix Comparison

In [ ]:
cls_comparison.plot_confusion_comparison()

## Per-Class F1 Comparison

In [ ]:
cls_comparison.plot_per_class_f1()

## Model Disagreement Analysis

In [ ]:
cls_comparison.model_disagreement_analysis()

## Summary

In [ ]:
print(f'\n' + '='*60)
print(f'MODEL COMPARISON ANALYSIS COMPLETE')
print(f'='*60)
print(f'\nAll visualizations saved to: {str_dirname_output}/')
print(f'  • 01_metrics_comparison.png')
print(f'  • 02_roc_comparison.png')
print(f'  • 03_confusion_comparison.png')
print(f'  • 04_per_class_f1.png')
print(f'\nBest overall model by Accuracy:')
int_best_idx = max(range(4), key=lambda i: cls_comparison.dict_metrics[i]['accuracy'])
print(f'  {cls_comparison.list_model_names[int_best_idx]}: {cls_comparison.dict_metrics[int_best_idx]["accuracy"]:.4f}')
print(f'\nBest overall model by F1-Macro:')
int_best_idx = max(range(4), key=lambda i: cls_comparison.dict_metrics[i]['f1_macro'])
print(f'  {cls_comparison.list_model_names[int_best_idx]}: {cls_comparison.dict_metrics[int_best_idx]["f1_macro"]:.4f}')